In [18]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import os

### Data Loading

In [19]:
PROCESSED_PATH = Path.cwd().parent.parent / "data" / "processed"
print(PROCESSED_PATH)
engineered_data = os.path.join(PROCESSED_PATH, "engineered_dataset.csv")
df = pd.read_csv(engineered_data)
df.head()

/home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed


,scenario_id,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,...,rolling_mean_velocity_b,rolling_std_velocity_b,rolling_mean_pressure_drop_bc,rolling_std_pressure_drop_bc,rolling_mean_midpoint_pressure_deviation,rolling_std_midpoint_pressure_deviation,rolling_mean_midpoint_velocity_deviation,rolling_std_midpoint_velocity_deviation,rolling_mean_midpoint_vof_deviation,rolling_std_midpoint_vof_deviation
0,blockage_25_loc16m,1,35793.499725,2.534141,0.034635,24.489446,11.197253,0.401012,19965.653843,2.474028,...,2.474028,0.000000,15575.231918,0.000000,-126.306983,0.000000,-0.056303,0.000000,-0.001483,0.000000
1,blockage_25_loc16m,2,36620.208522,2.484805,0.038275,25.658411,10.641124,0.405089,20589.416498,2.421521,...,2.447775,0.037128,15784.963909,296.605826,-72.177522,76.550617,-0.066276,0.014104,-0.002528,0.001478
2,blockage_25_loc16m,3,36251.794105,2.473708,0.034279,25.294662,9.484428,0.398871,20946.810911,2.531422,...,2.475657,0.054968,16232.488105,803.007614,255.640536,570.371841,-0.013167,0.092527,0.000671,0.005639
3,blockage_25_loc16m,4,35694.151655,2.478258,0.032199,27.022206,9.489620,0.393614,19637.008191,2.406712,...,2.458421,0.056592,16108.980877,700.640394,151.894868,509.838549,-0.032113,0.084518,0.002760,0.006216
4,blockage_25_loc16m,5,35834.383052,2.467021,0.037261,22.894842,10.660725,0.400522,19776.413014,2.442424,...,2.455221,0.049530,16234.319894,668.372701,189.286487,449.379767,-0.035610,0.073611,0.001922,0.005700


### Column definition

In [20]:
live_feature_cols = [
    "node_a_pressure", "velocity_a",
    "node_b_pressure", "velocity_b",
    "node_c_pressure", "velocity_c",

    "pressure_drop_ab", "pressure_drop_bc", "pressure_drop_ac",
    "pressure_gradient_ab", "pressure_gradient_bc", "pressure_gradient_ac",
    "expected_midpoint_pressure", "midpoint_pressure_deviation",
    "pressure_asymmetry",
    "pressure_ratio_ab", "pressure_ratio_bc", "pressure_ratio_ac",
    "upstream_downstream_ratio",
    "dp_dt_a", "dp_dt_b", "dp_dt_c",
    "d2p_dt2_a", "d2p_dt2_b", "d2p_dt2_c",

    "velocity_drop_ab", "velocity_drop_bc", "velocity_drop_ac",
    "mean_velocity",
    "velocity_deviation_a", "velocity_deviation_b", "velocity_deviation_c",
    "velocity_std",
    "expected_midpoint_velocity", "midpoint_velocity_deviation",
    "dv_dt_a", "dv_dt_b", "dv_dt_c",
    "d2v_dt2_a", "d2v_dt2_b", "d2v_dt2_c",

    "hydraulic_power_a", "hydraulic_power_b", "hydraulic_power_c",
    "power_loss_ab", "power_loss_bc", "power_loss_ac",
    "pv_ratio_b",
    "momentum_a", "momentum_b", "momentum_c",
    "momentum_loss_ab", "momentum_loss_bc",

    "leak_pressure_indicator",
    "blockage_pressure_indicator",
    "blockage_velocity_indicator",
    "leak_score",

    "rolling_mean_node_a_pressure", "rolling_std_node_a_pressure",
    "rolling_mean_node_b_pressure", "rolling_std_node_b_pressure",
    "rolling_mean_node_c_pressure", "rolling_std_node_c_pressure",
    "rolling_mean_velocity_b", "rolling_std_velocity_b",
    "rolling_mean_pressure_drop_bc", "rolling_std_pressure_drop_bc",
    "rolling_mean_midpoint_pressure_deviation",
    "rolling_std_midpoint_pressure_deviation",
    "rolling_mean_midpoint_velocity_deviation",
    "rolling_std_midpoint_velocity_deviation",
]

In [21]:
print(f"Full dataset shape   : {df.shape}")

missing = [f for f in live_feature_cols if f not in df.columns]
if missing:
    print(f"\n MISSING COLUMNS ({len(missing)}):")
    for col in missing:
        print(f"   - {col}")
else:
    print(f"All {len(live_feature_cols)} features found")

# Always keep scenario_id, timestep, label so sequences still work
meta_cols = ["scenario_id", "timestep", "label"]

keep_cols = meta_cols + live_feature_cols
df_live   = df[keep_cols].copy()

print(f"\nLive dataset shape   : {df_live.shape}")
print(f"Features             : {len(live_feature_cols)}")
print(f"Metadata cols        : {meta_cols}")

feature_cols = [c for c in df.columns
                if c not in ["scenario_id", "timestep", "label",
                             "fault_type", "effect_factor"]]

feature_path = Path.cwd().parent.parent / "artifacts" / "feature_names_live.txt"
feature_path.parent.mkdir(parents=True, exist_ok=True)

with open(feature_path, "w") as f:
    for feat in feature_cols:
        f.write(feat + "\n")

print(f"Saved {len(feature_cols)} features to: {feature_path}")
print(feature_cols)
save_path = PROCESSED_PATH /"live_feature_dataset.csv"
df_live.to_csv(save_path, index=False)

print(f"\nSaved → {save_path}")

df_check = pd.read_csv(save_path)
print(f"Reload shape         : {df_check.shape}")
print(f"Columns              : {list(df_check.columns[:5])} ... {list(df_check.columns[-3:])}")

Full dataset shape   : (31500, 131)
All 71 features found

Live dataset shape   : (31500, 74)
Features             : 71
Metadata cols        : ['scenario_id', 'timestep', 'label']
Saved 126 features to: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/artifacts/feature_names_live.txt
['node_a_pressure', 'velocity_a', 'tke_a', 'tdr_a', 'wall_shear_a', 'tailings_vof_a', 'node_b_pressure', 'velocity_b', 'tke_b', 'tdr_b', 'wall_shear_b', 'tailings_vof_b', 'node_c_pressure', 'velocity_c', 'tke_c', 'tdr_c', 'wall_shear_c', 'tailings_vof_c', 'pressure_drop_ab', 'pressure_drop_bc', 'pressure_drop_ac', 'pressure_gradient_ratio', 'dp_dt_a', 'dp_dt_b', 'dp_dt_c', 'velocity_drop_ab', 'velocity_drop_bc', 'tke_drop_ab', 'tke_drop_bc', 'flow_velocity', 'pressure_gradient_ab', 'pressure_gradient_bc', 'pressure_gradient_ac', 'expected_midpoint_pressure', 'midpoint_pressure_deviation', 'pressure_asymmetry', 'pressure_ratio_ab', 'pressure_ratio_bc', 'pressure_ratio_ac', 'upstream_downs